# Классификация уровня ремонта по фото квартир

Задача: по фотографиям объявления определить бинарный класс ремонта.

Классы:
- `bad_renovation` - `Косметический`;
- `good_renovation` - `Евроремонт` и `Дизайнерский`.

`Без ремонта` не используем, потому что таких примеров мало.

В ноутбуке сравниваются три подхода:
1. ImageNet ResNet50 embeddings + mean pooling + MLP;
2. DINOv2 ViT-B/14 embeddings + mean pooling + MLP;
3. DINOv2 ViT-B/14 embeddings + Set Transformer + MLP.

## 1. Импорты

In [18]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

os.environ.setdefault("PIP_DISABLE_PIP_VERSION_CHECK", "1")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "zstandard", "wandb"])

nvidia_smi = shutil.which("nvidia-smi")
gpu_info = subprocess.run(
    [nvidia_smi, "--query-gpu=name,compute_cap", "--format=csv,noheader"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    check=True,
).stdout.strip()
print("nvidia-smi:", gpu_info)

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "--no-cache-dir",
    "--extra-index-url",
    "https://download.pytorch.org/whl/cu121",
    "torch==2.5.1+cu121",
    "torchvision==0.20.1+cu121",
])

subprocess.check_call([
    sys.executable,
    "-c",
    "import torch, torchvision, numpy, scipy, sklearn; print('smoke:', torch.__version__, torchvision.__version__, numpy.__version__, scipy.__version__, sklearn.__version__)",
])

nvidia-smi: Tesla P100-PCIE-16GB, 6.0
smoke: 2.5.1+cu121 0.20.1+cu121 2.4.6 1.16.3 1.6.1


0

In [19]:
import json
import random
import shutil
import tarfile
import time
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import zstandard as zstd
from IPython.display import display
from PIL import Image, ImageFile, UnidentifiedImageError
from tqdm.auto import tqdm

import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, TensorDataset
from torchvision import models, transforms
from torchvision.models import ResNet50_Weights

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
    precision_recall_fscore_support,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

import wandb

ImageFile.LOAD_TRUNCATED_IMAGES = True

## 2. Random seed, устройство и конфиг

In [20]:
seed = 42

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

device = torch.device("cuda")
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")

print("Device:", torch.cuda.get_device_name(0))

DATASET_SLUG = "beldmian/moscow-rent-cian-with-images"
CSV_FILENAME = "moscow_rent_all.csv"
ARCHIVE_FILENAME = "images_512_q60.tar.zst"

CLASS_NAMES = ["bad_renovation", "good_renovation"]
CLASS_TO_IDX = {name: i for i, name in enumerate(CLASS_NAMES)}
IDX_TO_CLASS = {i: name for name, i in CLASS_TO_IDX.items()}
RU_TO_CLASS = {
    "Косметический": "bad_renovation",
    "Евроремонт": "good_renovation",
    "Дизайнерский": "good_renovation",
}

IMAGE_SIZE = 224
MAX_IMAGES_PER_OFFER = 8
VAL_SIZE = 0.20
NUM_WORKERS = 4

RESNET_FEATURE_BATCH_SIZE = 128
DINO_FEATURE_BATCH_SIZE = 64
VECTOR_TRAIN_BATCH_SIZE = 512
SET_TRAIN_BATCH_SIZE = 256

VECTOR_EPOCHS = 45
SET_EPOCHS = 65
VECTOR_PATIENCE = 9
SET_PATIENCE = 12

WANDB_PROJECT = "cian-renovation-binary-architecture-comparison"
WANDB_GROUP = "binary_architecture_comparison_" + datetime.utcnow().strftime("%Y%m%d_%H%M%S")

WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
TEMP_DIR = Path("/kaggle/temp") if Path("/kaggle/temp").exists() else Path("tmp")
INPUT_DIR = Path("/kaggle/input") if Path("/kaggle/input").exists() else Path.cwd()

ARTIFACT_DIR = WORK_DIR / "architecture_comparison_artifacts"
IMAGE_ROOT = TEMP_DIR / "binary_architecture_selected_images"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_ROOT.mkdir(parents=True, exist_ok=True)

device

Device: Tesla P100-PCIE-16GB


/tmp/ipykernel_58/1538199058.py:43: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  WANDB_GROUP = "binary_architecture_comparison_" + datetime.utcnow().strftime("%Y%m%d_%H%M%S")


device(type='cuda')

## 3. W&B

In [21]:
wandb.login()

True

## 4. Загрузка данных

In [22]:
def find_input_file(filename: str) -> Path:
    matches = sorted(INPUT_DIR.rglob(filename))
    if not matches:
        raise FileNotFoundError(f"Не найден {filename} внутри {INPUT_DIR}")
    return matches[0]

CSV_PATH = find_input_file(CSV_FILENAME)
ARCHIVE_PATH = find_input_file(ARCHIVE_FILENAME)

print("CSV:", CSV_PATH)
print("Archive:", ARCHIVE_PATH)


CSV: /kaggle/input/datasets/beldmian/moscow-rent-cian-with-images/moscow_rent_all.csv
Archive: /kaggle/input/datasets/beldmian/moscow-rent-cian-with-images/images_512_q60.tar.zst


In [23]:
def renovation_from_detail_features(value: str) -> str | None:
    if not isinstance(value, str) or not value.strip():
        return None
    try:
        label = json.loads(value).get("Ремонт")
    except json.JSONDecodeError:
        return None
    return str(label).strip() if label else None


def split_image_paths(value: str) -> list[str]:
    if not isinstance(value, str) or not value.strip():
        return []
    return [part.strip().lstrip("/") for part in value.split("|") if part.strip()]


raw_df = pd.read_csv(
    CSV_PATH,
    sep=";",
    usecols=["offer_id", "detail_features", "image_paths"],
    dtype=str,
    low_memory=False,
)

raw_df["renovation_ru"] = raw_df["detail_features"].map(renovation_from_detail_features)
raw_df["class_name"] = raw_df["renovation_ru"].map(RU_TO_CLASS)
raw_df["path_list"] = raw_df["image_paths"].map(split_image_paths)
raw_df["n_images"] = raw_df["path_list"].map(len)

labeled_df = raw_df[raw_df["class_name"].notna() & (raw_df["n_images"] > 0)].copy()
labeled_df["offer_id"] = labeled_df["offer_id"].astype(str)
labeled_df = labeled_df.drop_duplicates("offer_id").reset_index(drop=True)
labeled_df["label"] = labeled_df["class_name"].map(CLASS_TO_IDX).astype(int)
labeled_df["selected_paths"] = labeled_df["path_list"].map(lambda paths: paths[:MAX_IMAGES_PER_OFFER])

print("Всего строк:", len(raw_df))
print("Объявлений с целевым классом и фото:", len(labeled_df))
display(labeled_df["renovation_ru"].value_counts().to_frame("offers"))
display(labeled_df["class_name"].value_counts().reindex(CLASS_NAMES, fill_value=0).to_frame("offers"))


Всего строк: 24055
Объявлений с целевым классом и фото: 20065


,offers
renovation_ru,
Евроремонт,7879
Дизайнерский,6289
Косметический,5897


,offers
class_name,
bad_renovation,5897
good_renovation,14168


## 5. Извлечение изображений

In [24]:
wanted_paths = sorted({path for paths in labeled_df["selected_paths"] for path in paths})

def extract_selected_images(archive_path: Path, output_dir: Path, wanted: set[str]) -> int:
    extracted = 0
    with archive_path.open("rb") as compressed_file:
        stream = zstd.ZstdDecompressor().stream_reader(compressed_file)
        with tarfile.open(fileobj=stream, mode="r|") as tar:
            for member in tqdm(tar, desc="extract selected images"):
                name = member.name.lstrip("./").lstrip("/")
                if name not in wanted or not member.isfile():
                    continue
                if ".." in Path(name).parts:
                    raise ValueError(f"Unsafe archive path: {name}")

                output_path = output_dir / name
                output_path.parent.mkdir(parents=True, exist_ok=True)
                source = tar.extractfile(member)
                if source is not None:
                    with output_path.open("wb") as destination:
                        shutil.copyfileobj(source, destination)
                    extracted += 1
    return extracted

missing_paths = [path for path in wanted_paths if not (IMAGE_ROOT / path).exists()]

print(extract_selected_images(ARCHIVE_PATH, IMAGE_ROOT, set(missing_paths)))

labeled_df["available_paths"] = labeled_df["selected_paths"].map(
    lambda paths: [path for path in paths if (IMAGE_ROOT / path).exists()]
)
labeled_df = labeled_df[labeled_df["available_paths"].map(len) > 0].reset_index(drop=True)

display(labeled_df["available_paths"].map(len).describe().to_frame("n_available_images"))

extract selected images: 0it [00:00, ?it/s]

0


,n_available_images
count,20065.000000
mean,7.895739
std,0.485017
min,1.000000
25%,8.000000
50%,8.000000
75%,8.000000
max,8.000000


## 6. Train / Validation split

In [25]:
split_columns = ["offer_id", "renovation_ru", "class_name", "label", "available_paths"]

train_df, val_df = train_test_split(
    labeled_df[split_columns],
    test_size=VAL_SIZE,
    random_state=seed,
    stratify=labeled_df["label"],
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

summary_counts = pd.DataFrame({
    "train": train_df["class_name"].value_counts(),
    "validation": val_df["class_name"].value_counts(),
}).reindex(CLASS_NAMES).fillna(0).astype(int)

display(summary_counts)

split_df = pd.concat([
    train_df.assign(split="train"),
    val_df.assign(split="validation"),
], ignore_index=True)
split_df[["offer_id", "split", "renovation_ru", "class_name", "label"]].to_csv(
    ARTIFACT_DIR / "train_validation_split.csv",
    index=False,
)

,train,validation
class_name,,
bad_renovation,4718,1179
good_renovation,11334,2834


## 7. Dataset и извлечение эмбеддингов

In [26]:
image_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class ImageTaskDataset(Dataset):
    def __init__(self, offers: pd.DataFrame, transform):
        self.transform = transform
        self.tasks = [
            (offer_index, slot, rel_path)
            for offer_index, paths in enumerate(offers["available_paths"])
            for slot, rel_path in enumerate(paths[:MAX_IMAGES_PER_OFFER])
        ]

    def __len__(self) -> int:
        return len(self.tasks)

    def __getitem__(self, index: int):
        offer_index, slot, rel_path = self.tasks[index]
        try:
            image = Image.open(IMAGE_ROOT / rel_path).convert("RGB")
        except (UnidentifiedImageError, OSError):
            image = Image.new("RGB", (IMAGE_SIZE, IMAGE_SIZE), color=(0, 0, 0))
        return self.transform(image), offer_index, slot

def masked_mean_pool(embeddings: np.ndarray, masks: np.ndarray) -> np.ndarray:
    mask = masks.astype(np.float32)[..., None]
    return (embeddings.astype(np.float32) * mask).sum(1) / mask.sum(1).clip(min=1.0)

def extract_offer_features(offers: pd.DataFrame, encoder: nn.Module, feature_fn, batch_size: int, name: str) -> dict:
    dataset = ImageTaskDataset(offers, image_transform)
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=NUM_WORKERS > 0,
        prefetch_factor=2 if NUM_WORKERS > 0 else None,
    )

    labels = offers["label"].to_numpy(dtype=np.int64)
    offer_ids = offers["offer_id"].astype(str).to_numpy()
    masks = np.zeros((len(offers), MAX_IMAGES_PER_OFFER), dtype=bool)
    embeddings = None
    start = time.time()

    encoder.eval()
    for images, offer_indices, slots in tqdm(loader, desc=f"extract {name} features"):
        images = images.to(device, non_blocking=True)
        with torch.inference_mode(), torch.autocast(device_type="cuda", dtype=torch.float16):
            features = feature_fn(encoder, images).float()

        features_np = features.cpu().numpy().astype(np.float16)
        if embeddings is None:
            embeddings = np.zeros((len(offers), MAX_IMAGES_PER_OFFER, features_np.shape[1]), dtype=np.float16)

        offer_indices = offer_indices.numpy()
        slots = slots.numpy()
        embeddings[offer_indices, slots] = features_np
        masks[offer_indices, slots] = True

    elapsed = time.time() - start
    print(f"{name} embeddings:", embeddings.shape, "seconds:", round(elapsed, 1), "images/sec:", round(len(dataset) / elapsed, 1))
    return {
        "embeddings": embeddings,
        "masks": masks,
        "labels": labels,
        "offer_ids": offer_ids,
        "feature_dim": int(embeddings.shape[-1]),
        "seconds": elapsed,
    }

## 8. Метрики

In [27]:
def safe_score(metric_fn, *args, default=np.nan, **kwargs) -> float:
    try:
        return float(metric_fn(*args, **kwargs))
    except ValueError:
        return float(default)

def calculate_metrics(y_true: np.ndarray, probs: np.ndarray) -> dict:
    probs = np.clip(probs.astype(np.float64), 1e-7, 1.0 - 1e-7)
    probs = probs / probs.sum(axis=1, keepdims=True)
    y_pred = probs.argmax(axis=1)

    labels = np.arange(len(CLASS_NAMES))
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=labels,
        zero_division=0,
    )

    good_index = CLASS_TO_IDX["good_renovation"]
    y_good = (y_true == good_index).astype(int)

    per_class = [
        {
            "class_name": CLASS_NAMES[i],
            "precision": float(precision[i]),
            "recall": float(recall[i]),
            "f1": float(f1[i]),
            "support": int(support[i]),
        }
        for i in labels
    ]

    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1.mean()),
        "weighted_f1": float(np.average(f1, weights=support)),
        "balanced_accuracy": float(recall.mean()),
        "log_loss": safe_score(log_loss, y_true, probs, labels=labels.tolist()),
        "brier_good": safe_score(brier_score_loss, y_good, probs[:, good_index]),
        "roc_auc_good": safe_score(roc_auc_score, y_good, probs[:, good_index]),
        "average_precision_good": safe_score(average_precision_score, y_good, probs[:, good_index]),
        "prediction_bad_count": int((y_pred == CLASS_TO_IDX["bad_renovation"]).sum()),
        "prediction_good_count": int((y_pred == good_index).sum()),
        "per_class": per_class,
        "predictions": y_pred,
        "scores": probs,
    }

def wandb_metrics(prefix: str, metrics: dict) -> dict:
    keys = [
        "accuracy",
        "macro_f1",
        "weighted_f1",
        "balanced_accuracy",
        "log_loss",
        "brier_good",
        "roc_auc_good",
        "average_precision_good",
    ]
    return {f"{prefix}/{key}": metrics[key] for key in keys}

def prediction_table(offer_ids, y_true, metrics) -> pd.DataFrame:
    return pd.DataFrame({
        "offer_id": offer_ids,
        "true_label": y_true.astype(int),
        "true_class": [IDX_TO_CLASS[int(y)] for y in y_true],
        "pred_label": metrics["predictions"].astype(int),
        "pred_class": [IDX_TO_CLASS[int(y)] for y in metrics["predictions"]],
        "p_bad_renovation": metrics["scores"][:, 0],
        "p_good_renovation": metrics["scores"][:, 1],
    })


## 9. Архитектуры моделей

In [28]:
class FeatureMLP(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int = 512, dropout: float = 0.25):
        super().__init__()
        self.model = nn.Sequential(
            nn.LayerNorm(input_dim),
            nn.Linear(input_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, len(CLASS_NAMES)),
        )

    def forward(self, x):
        return self.model(x)

class SetFeatureDataset(Dataset):
    def __init__(self, embeddings, masks, labels, image_dropout: float = 0.0):
        self.embeddings = embeddings
        self.masks = masks
        self.labels = labels
        self.image_dropout = image_dropout

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        x = torch.from_numpy(self.embeddings[index].astype(np.float32))
        mask = torch.from_numpy(self.masks[index].astype(bool))
        y = int(self.labels[index])

        if self.image_dropout > 0:
            valid = torch.where(mask)[0]
            drop = torch.rand(len(valid)) < self.image_dropout
            if len(valid) > 1 and not bool(drop.all()):
                mask[valid[drop]] = False
                x[~mask] = 0.0

        return x, mask, y

class SABlock(nn.Module):
    def __init__(self, dim: int, heads: int, dropout: float):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, heads, dropout=dropout, batch_first=True)
        self.drop = nn.Dropout(dropout)
        self.norm2 = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(
            nn.Linear(dim, dim * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim * 4, dim),
            nn.Dropout(dropout),
        )

    def forward(self, x, mask):
        h = self.norm1(x)
        attn_out, _ = self.attn(h, h, h, key_padding_mask=~mask, need_weights=False)
        x = x + self.drop(attn_out)
        x = x + self.ffn(self.norm2(x))
        return x.masked_fill(~mask.unsqueeze(-1), 0.0)

class SetTransformerClassifier(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int = 384, heads: int = 8, depth: int = 4, pma_seeds: int = 4, dropout: float = 0.20):
        super().__init__()
        self.slot_embeddings = nn.Parameter(torch.randn(1, MAX_IMAGES_PER_OFFER, input_dim) * 0.02)
        self.input_projection = nn.Sequential(
            nn.LayerNorm(input_dim),
            nn.Linear(input_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.blocks = nn.ModuleList([SABlock(hidden_dim, heads, dropout) for _ in range(depth)])
        self.seed_vectors = nn.Parameter(torch.randn(1, pma_seeds, hidden_dim) * 0.02)
        self.pma = nn.MultiheadAttention(hidden_dim, heads, dropout=dropout, batch_first=True)
        self.pma_norm = nn.LayerNorm(hidden_dim)
        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_dim * (pma_seeds + 2)),
            nn.Linear(hidden_dim * (pma_seeds + 2), hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, len(CLASS_NAMES)),
        )

    def forward(self, x, mask):
        x = x + self.slot_embeddings[:, : x.shape[1], :]
        h = self.input_projection(x).masked_fill(~mask.unsqueeze(-1), 0.0)
        for block in self.blocks:
            h = block(h, mask)

        seeds = self.seed_vectors.expand(h.shape[0], -1, -1)
        pma_out, _ = self.pma(seeds, h, h, key_padding_mask=~mask, need_weights=False)
        pma_out = self.pma_norm(pma_out + seeds).flatten(1)

        mask_f = mask.unsqueeze(-1).float()
        mean_pool = (h * mask_f).sum(1) / mask_f.sum(1).clamp_min(1.0)
        max_pool = h.masked_fill(~mask.unsqueeze(-1), -1e4).max(1).values
        return self.classifier(torch.cat([pma_out, mean_pool, max_pool], dim=1))

## 10. Функция обучения

In [29]:
def class_weights_tensor(labels: np.ndarray) -> torch.Tensor:
    counts = np.bincount(labels, minlength=len(CLASS_NAMES)).astype(np.float32)
    weights = len(labels) / (len(CLASS_NAMES) * counts)
    return torch.tensor(weights / weights.mean(), dtype=torch.float32, device=device)

def make_vector_loaders(train_pack: dict, val_pack: dict):
    X_train = masked_mean_pool(train_pack["embeddings"], train_pack["masks"])
    X_val = masked_mean_pool(val_pack["embeddings"], val_pack["masks"])
    y_train = train_pack["labels"]
    y_val = val_pack["labels"]

    train_dataset = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.long))
    val_dataset = TensorDataset(torch.tensor(X_val, dtype=torch.float32), torch.tensor(y_val, dtype=torch.long))

    train_loader = DataLoader(train_dataset, batch_size=VECTOR_TRAIN_BATCH_SIZE, shuffle=True, pin_memory=device.type == "cuda")
    val_loader = DataLoader(val_dataset, batch_size=VECTOR_TRAIN_BATCH_SIZE * 2, shuffle=False, pin_memory=device.type == "cuda")
    return train_loader, val_loader, y_train

def make_set_loaders(train_pack: dict, val_pack: dict, image_dropout: float = 0.18):
    y_train = train_pack["labels"]
    y_val = val_pack["labels"]
    train_dataset = SetFeatureDataset(train_pack["embeddings"], train_pack["masks"], y_train, image_dropout=image_dropout)
    val_dataset = SetFeatureDataset(val_pack["embeddings"], val_pack["masks"], y_val)

    train_loader = DataLoader(train_dataset, batch_size=SET_TRAIN_BATCH_SIZE, shuffle=True, pin_memory=device.type == "cuda")
    val_loader = DataLoader(val_dataset, batch_size=SET_TRAIN_BATCH_SIZE * 2, shuffle=False, pin_memory=device.type == "cuda")
    return train_loader, val_loader, y_train

def batch_to_device(batch):
    if len(batch) == 2:
        x, y = batch
        return (x.to(device),), y.to(device)

    x, mask, y = batch
    return (x.to(device), mask.to(device)), y.to(device)

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total = 0

    for batch in loader:
        inputs, y = batch_to_device(batch)
        optimizer.zero_grad()

        logits = model(*inputs)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        batch_size = y.size(0)
        total_loss += float(loss.detach().cpu()) * batch_size
        total_correct += int((logits.argmax(1) == y).sum())
        total += batch_size

    return total_loss / total, total_correct / total

@torch.no_grad()
def predict(model, loader):
    model.eval()
    probs = []
    labels = []

    for batch in loader:
        inputs, y = batch_to_device(batch)
        logits = model(*inputs)
        probs.append(torch.softmax(logits, dim=1).cpu().numpy())
        labels.append(y.cpu().numpy())

    return np.concatenate(probs), np.concatenate(labels)

def train_model(run_name: str, model: nn.Module, train_loader: DataLoader, val_loader: DataLoader, val_offer_ids: np.ndarray, train_labels: np.ndarray, config: dict):
    run = wandb.init(
        project=WANDB_PROJECT,
        group=WANDB_GROUP,
        name=run_name,
        config=config,
        reinit="finish_previous",
    )

    model = model.to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor(train_labels), label_smoothing=0.02)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config["epochs"])

    best_state = None
    best_epoch = 0
    best_macro_f1 = -1.0
    patience_left = config["patience"]
    history = []

    for epoch in range(1, config["epochs"] + 1):
        train_loss, train_accuracy = train_epoch(model, train_loader, optimizer, criterion)
        scheduler.step()

        val_probs, y_val = predict(model, val_loader)
        val_metrics = calculate_metrics(y_val, val_probs)

        row = {
            "epoch": epoch,
            "lr": optimizer.param_groups[0]["lr"],
            "train/loss": train_loss,
            "train/accuracy": train_accuracy,
            **wandb_metrics("val", val_metrics),
        }
        history.append(row)
        wandb.log(row, step=epoch)

        print(
            f"{run_name} | epoch {epoch:03d} | "
            f"loss={train_loss:.4f} | "
            f"val_f1={val_metrics['macro_f1']:.4f} | "
            f"val_acc={val_metrics['accuracy']:.4f}"
        )

        if val_metrics["macro_f1"] > best_macro_f1:
            best_macro_f1 = val_metrics["macro_f1"]
            best_epoch = epoch
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            patience_left = config["patience"]
        else:
            patience_left -= 1
            if patience_left == 0:
                print(f"Early stopping at epoch {epoch}")
                break

    model.load_state_dict(best_state)
    val_probs, y_val = predict(model, val_loader)
    metrics = calculate_metrics(y_val, val_probs)

    checkpoint_path = ARTIFACT_DIR / f"{run_name}.pt"
    history_path = ARTIFACT_DIR / f"{run_name}_history.csv"
    predictions_path = ARTIFACT_DIR / f"{run_name}_validation_predictions.csv"

    torch.save({
        "model_state_dict": best_state,
        "model_config": config,
        "classes": CLASS_NAMES,
        "class_to_idx": CLASS_TO_IDX,
        "best_epoch": best_epoch,
        "metrics": {k: v for k, v in metrics.items() if k not in ["predictions", "scores"]},
    }, checkpoint_path)
    pd.DataFrame(history).to_csv(history_path, index=False)
    prediction_table(val_offer_ids, y_val, metrics).to_csv(predictions_path, index=False)

    wandb.summary["best_epoch"] = best_epoch
    wandb.summary["best_val_macro_f1"] = best_macro_f1
    wandb.summary.update(wandb_metrics("final_val", metrics))
    run.finish()

    return {
        "run_name": run_name,
        "architecture_family": config["architecture_family"],
        "best_epoch": best_epoch,
        "metrics": {k: v for k, v in metrics.items() if k not in ["predictions", "scores"]},
        "checkpoint_path": str(checkpoint_path),
        "history_path": str(history_path),
        "predictions_path": str(predictions_path),
    }

## 11. ImageNet ResNet50 + Mean Pooling + MLP

In [30]:
results = []

resnet = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
resnet_encoder = nn.Sequential(*list(resnet.children())[:-1]).to(device).eval()
for parameter in resnet_encoder.parameters():
    parameter.requires_grad_(False)

@torch.no_grad()
def resnet_features(encoder, images):
    return F.normalize(encoder(images).flatten(1), p=2, dim=1)

resnet_train = extract_offer_features(train_df, resnet_encoder, resnet_features, RESNET_FEATURE_BATCH_SIZE, "resnet50_train")
resnet_val = extract_offer_features(val_df, resnet_encoder, resnet_features, RESNET_FEATURE_BATCH_SIZE, "resnet50_validation")
train_loader, val_loader, y_train = make_vector_loaders(resnet_train, resnet_val)

results.append(train_model(
    run_name="01_resnet50_meanpool_mlp",
    model=FeatureMLP(input_dim=resnet_train["feature_dim"], hidden_dim=512, dropout=0.25),
    train_loader=train_loader,
    val_loader=val_loader,
    val_offer_ids=resnet_val["offer_ids"],
    train_labels=y_train,
    config={
        "architecture_family": "resnet_meanpool_mlp",
        "feature_extractor": "torchvision resnet50 IMAGENET1K_V2",
        "feature_dim": resnet_train["feature_dim"],
        "pooling": "masked_mean_pooling_over_images",
        "classifier": "MLP",
        "epochs": VECTOR_EPOCHS,
        "patience": VECTOR_PATIENCE,
        "lr": 3e-4,
        "weight_decay": 2e-4,
    },
))

del resnet, resnet_encoder, resnet_train, resnet_val, train_loader, val_loader
torch.cuda.empty_cache()

extract resnet50_train features:   0%|          | 0/991 [00:00<?, ?it/s]

resnet50_train embeddings: (16052, 8, 2048) seconds: 310.7 images/sec: 408.1


extract resnet50_validation features:   0%|          | 0/248 [00:00<?, ?it/s]

resnet50_validation embeddings: (4013, 8, 2048) seconds: 80.3 images/sec: 394.2


01_resnet50_meanpool_mlp | epoch 001 | loss=0.5783 | val_f1=0.7041 | val_acc=0.7222
01_resnet50_meanpool_mlp | epoch 002 | loss=0.5143 | val_f1=0.7513 | val_acc=0.7830
01_resnet50_meanpool_mlp | epoch 003 | loss=0.4955 | val_f1=0.6958 | val_acc=0.7082
01_resnet50_meanpool_mlp | epoch 004 | loss=0.4894 | val_f1=0.7507 | val_acc=0.7790
01_resnet50_meanpool_mlp | epoch 005 | loss=0.4787 | val_f1=0.7625 | val_acc=0.7952
01_resnet50_meanpool_mlp | epoch 006 | loss=0.4691 | val_f1=0.7098 | val_acc=0.7239
01_resnet50_meanpool_mlp | epoch 007 | loss=0.4647 | val_f1=0.7471 | val_acc=0.7735
01_resnet50_meanpool_mlp | epoch 008 | loss=0.4582 | val_f1=0.7459 | val_acc=0.7735
01_resnet50_meanpool_mlp | epoch 009 | loss=0.4473 | val_f1=0.7522 | val_acc=0.7800
01_resnet50_meanpool_mlp | epoch 010 | loss=0.4395 | val_f1=0.7605 | val_acc=0.7929
01_resnet50_meanpool_mlp | epoch 011 | loss=0.4286 | val_f1=0.7649 | val_acc=0.8036
01_resnet50_meanpool_mlp | epoch 012 | loss=0.4261 | val_f1=0.7435 | val_acc

epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
lr,█████▇▇▇▆▆▆▅▅▄▄▃▃▂▂▁
train/accuracy,▁▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇██
train/loss,█▆▆▆▅▅▅▅▄▄▄▄▃▃▃▂▂▂▁▁
val/accuracy,▂▆▁▆▇▂▆▆▆▇█▅▇▇▇▆▃▄▃█
val/average_precision_good,▁▅▆▇██▇▇███▇█▇▅▅▄▃▂▂
val/balanced_accuracy,▂▆▄▇█▆▇▇██▆██▇▅▇▄▅▃▁
val/brier_good,▇▃█▃▂▇▃▃▃▂▁▄▃▂▂▃▆▅▆▂
val/log_loss,▆▃█▃▂▇▃▃▃▂▁▄▃▂▂▄▇▆▇▃
val/macro_f1,▂▇▁▇█▂▆▆▇██▆▇█▇▆▃▄▃▇
+2,...


## 12. DINOv2 эмбеддинги

In [31]:
dino = torch.hub.load("facebookresearch/dinov2", "dinov2_vitb14_reg", pretrained=True, trust_repo=True).to(device).eval()
for parameter in dino.parameters():
    parameter.requires_grad_(False)

@torch.no_grad()
def dino_features(encoder, images):
    output = encoder(images)
    features = output["x_norm_clstoken"] if isinstance(output, dict) else output
    return F.normalize(features.float(), p=2, dim=1)

dino_train = extract_offer_features(train_df, dino, dino_features, DINO_FEATURE_BATCH_SIZE, "dinov2_vitb14_train")
dino_val = extract_offer_features(val_df, dino, dino_features, DINO_FEATURE_BATCH_SIZE, "dinov2_vitb14_validation")

del dino
torch.cuda.empty_cache()

Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main


extract dinov2_vitb14_train features:   0%|          | 0/1981 [00:00<?, ?it/s]

dinov2_vitb14_train embeddings: (16052, 8, 768) seconds: 959.1 images/sec: 132.2


extract dinov2_vitb14_validation features:   0%|          | 0/495 [00:00<?, ?it/s]

dinov2_vitb14_validation embeddings: (4013, 8, 768) seconds: 241.2 images/sec: 131.2


## 13. DINOv2 + Mean Pooling + MLP

In [32]:
train_loader, val_loader, y_train = make_vector_loaders(dino_train, dino_val)

results.append(train_model(
    run_name="02_dinov2_vitb_meanpool_mlp",
    model=FeatureMLP(input_dim=dino_train["feature_dim"], hidden_dim=512, dropout=0.25),
    train_loader=train_loader,
    val_loader=val_loader,
    val_offer_ids=dino_val["offer_ids"],
    train_labels=y_train,
    config={
        "architecture_family": "dino_meanpool_mlp",
        "feature_extractor": "dinov2_vitb14_reg",
        "feature_dim": dino_train["feature_dim"],
        "pooling": "masked_mean_pooling_over_images",
        "classifier": "MLP",
        "epochs": VECTOR_EPOCHS,
        "patience": VECTOR_PATIENCE,
        "lr": 3e-4,
        "weight_decay": 2e-4,
    },
))

del train_loader, val_loader
torch.cuda.empty_cache()

02_dinov2_vitb_meanpool_mlp | epoch 001 | loss=0.5230 | val_f1=0.7567 | val_acc=0.7835
02_dinov2_vitb_meanpool_mlp | epoch 002 | loss=0.4747 | val_f1=0.7699 | val_acc=0.8036
02_dinov2_vitb_meanpool_mlp | epoch 003 | loss=0.4654 | val_f1=0.7500 | val_acc=0.7732
02_dinov2_vitb_meanpool_mlp | epoch 004 | loss=0.4594 | val_f1=0.7720 | val_acc=0.8071
02_dinov2_vitb_meanpool_mlp | epoch 005 | loss=0.4540 | val_f1=0.7620 | val_acc=0.7889
02_dinov2_vitb_meanpool_mlp | epoch 006 | loss=0.4427 | val_f1=0.7609 | val_acc=0.7872
02_dinov2_vitb_meanpool_mlp | epoch 007 | loss=0.4334 | val_f1=0.7669 | val_acc=0.7967
02_dinov2_vitb_meanpool_mlp | epoch 008 | loss=0.4228 | val_f1=0.7630 | val_acc=0.7947
02_dinov2_vitb_meanpool_mlp | epoch 009 | loss=0.4129 | val_f1=0.7525 | val_acc=0.7775
02_dinov2_vitb_meanpool_mlp | epoch 010 | loss=0.3997 | val_f1=0.7638 | val_acc=0.7927
02_dinov2_vitb_meanpool_mlp | epoch 011 | loss=0.3818 | val_f1=0.7494 | val_acc=0.7737
02_dinov2_vitb_meanpool_mlp | epoch 012 | l

epoch,▁▂▂▃▃▄▅▅▆▆▇▇█
lr,███▇▇▆▆▅▅▄▃▂▁
train/accuracy,▁▄▄▄▄▅▅▅▆▆▇▇█
train/loss,█▆▆▆▅▅▄▄▄▃▂▂▁
val/accuracy,▃▇▁█▄▄▆▅▂▅▁▆▃
val/average_precision_good,▅▆▇████▆▇▆▅▂▁
val/balanced_accuracy,▆▆▇▆███▅▆▇▅▁▃
val/brier_good,▆▂█▁▄▅▃▃▇▄█▄▆
val/log_loss,▆▂▇▁▄▅▃▃▇▄█▄▇
val/macro_f1,▃▇▁█▅▅▆▅▂▅▁▄▂
+2,...


## 14. DINOv2 + Set Transformer + MLP

In [33]:
train_loader, val_loader, y_train = make_set_loaders(dino_train, dino_val, image_dropout=0.18)

results.append(train_model(
    run_name="03_dinov2_vitb_set_transformer",
    model=SetTransformerClassifier(
        input_dim=dino_train["feature_dim"],
        hidden_dim=384,
        heads=8,
        depth=4,
        pma_seeds=4,
        dropout=0.20,
    ),
    train_loader=train_loader,
    val_loader=val_loader,
    val_offer_ids=dino_val["offer_ids"],
    train_labels=y_train,
    config={
        "architecture_family": "dino_set_transformer",
        "feature_extractor": "dinov2_vitb14_reg",
        "feature_dim": dino_train["feature_dim"],
        "set_encoder": "Set Transformer, 4 SAB blocks, 8 heads, PMA seeds=4, mean+max pooling",
        "classifier": "MLP",
        "epochs": SET_EPOCHS,
        "patience": SET_PATIENCE,
        "lr": 2e-4,
        "weight_decay": 2e-4,
    },
))

torch.cuda.empty_cache()

03_dinov2_vitb_set_transformer | epoch 001 | loss=0.5368 | val_f1=0.7531 | val_acc=0.7782
03_dinov2_vitb_set_transformer | epoch 002 | loss=0.4870 | val_f1=0.7470 | val_acc=0.7692
03_dinov2_vitb_set_transformer | epoch 003 | loss=0.4678 | val_f1=0.7116 | val_acc=0.7254
03_dinov2_vitb_set_transformer | epoch 004 | loss=0.4654 | val_f1=0.7386 | val_acc=0.7580
03_dinov2_vitb_set_transformer | epoch 005 | loss=0.4503 | val_f1=0.7440 | val_acc=0.7673
03_dinov2_vitb_set_transformer | epoch 006 | loss=0.4423 | val_f1=0.7335 | val_acc=0.7526
03_dinov2_vitb_set_transformer | epoch 007 | loss=0.4338 | val_f1=0.7744 | val_acc=0.8149
03_dinov2_vitb_set_transformer | epoch 008 | loss=0.4206 | val_f1=0.7615 | val_acc=0.7919
03_dinov2_vitb_set_transformer | epoch 009 | loss=0.4140 | val_f1=0.7680 | val_acc=0.8026
03_dinov2_vitb_set_transformer | epoch 010 | loss=0.3978 | val_f1=0.7601 | val_acc=0.7912
03_dinov2_vitb_set_transformer | epoch 011 | loss=0.3789 | val_f1=0.7473 | val_acc=0.7732
03_dinov2_

epoch,▁▁▂▂▃▃▃▄▄▅▅▅▆▆▆▇▇██
lr,█████▇▇▇▆▆▆▅▅▄▄▃▂▂▁
train/accuracy,▁▂▃▃▃▃▃▄▄▄▅▅▆▆▇▇▇██
train/loss,█▇▆▆▆▆▅▅▅▅▄▄▃▃▂▂▂▁▁
val/accuracy,▅▄▁▄▄▃█▆▇▆▅▅▇▆▃▅▇▅▅
val/average_precision_good,▇█████▇▇▆▆▆▆▁▁▆▃▄▂▄
val/balanced_accuracy,██▄█▇▆▆▇▇▇▆▇▆▁▁▃▂▂▁
val/brier_good,▄▃▇▅▄▅▁▃▂▄▅▄▄▄█▅▄▆▆
val/log_loss,▃▃▅▄▃▄▁▂▂▄▅▃▄▄█▄▄▇▆
val/macro_f1,▆▅▁▄▅▃█▇▇▆▅▆▇▅▂▅▆▅▅
+2,...


## 15. Сравнение моделей

In [34]:
comparison_rows = []

for item in results:
    metrics = item["metrics"]
    comparison_rows.append({
        "run_name": item["run_name"],
        "best_epoch": item["best_epoch"],
        "validation_accuracy": metrics["accuracy"],
        "validation_macro_f1": metrics["macro_f1"],
        "validation_weighted_f1": metrics["weighted_f1"],
        "validation_balanced_accuracy": metrics["balanced_accuracy"],
        "validation_roc_auc_good": metrics["roc_auc_good"],
        "validation_average_precision_good": metrics["average_precision_good"],
        "validation_bad_f1": metrics["per_class"][CLASS_TO_IDX["bad_renovation"]]["f1"],
        "validation_good_f1": metrics["per_class"][CLASS_TO_IDX["good_renovation"]]["f1"],
    })

comparison_df = pd.DataFrame(comparison_rows).sort_values("validation_macro_f1", ascending=False).reset_index(drop=True)
display(comparison_df)

comparison_df.to_csv(ARTIFACT_DIR / "architecture_comparison_metrics.csv", index=False)
(ARTIFACT_DIR / "architecture_comparison_results.json").write_text(
    json.dumps(results, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

,run_name,best_epoch,validation_accuracy,validation_macro_f1,validation_weighted_f1,validation_balanced_accuracy,validation_roc_auc_good,validation_average_precision_good,validation_bad_f1,validation_good_f1
0,03_dinov2_vitb_set_transformer,7,0.814852,0.774359,0.813780,0.771335,0.854910,0.920071,0.678772,0.869946
1,02_dinov2_vitb_meanpool_mlp,4,0.807127,0.772008,0.808911,0.777506,0.861607,0.928504,0.682527,0.861489
2,01_resnet50_meanpool_mlp,11,0.803638,0.764898,0.804256,0.766616,0.852317,0.921696,0.669463,0.860333


4072